# NB 4.3 &mdash; Demà plourà? Mesurar els errors d'un classificador

**MP 5134** &mdash; UT4

*Dades: AEMET, estació de l'aeroport de Palma (2015-2023).*

---
### Què farem avui

Fins ara hem mesurat els classificadors amb un sol número, el percentatge d'encerts.
Avui farem un classificador per a un problema real, **saber si demà plourà a Palma**, i
veurem que aquest número, sol, ens pot enganyar del tot. Aprendrem a mirar els errors un
per un i a triar el model segons qui l'ha de fer servir.

Com als notebooks anteriors, hi ha tres pauses «Ara tu» llargues per fer en grup.

Quan acabis hauries de saber respondre aquestes preguntes:

- Què és la matriu de confusió i com es llegeix?
- Què responen la precisió i el *recall*, en paraules de cada dia?
- Per què un model que encerta el 87% dels dies pot no servir de res?
- Com es tria el llindar segons qui farà servir el model?
- Què diu la corba ROC?

## 1. Les dades: nou anys de temps a Palma

L'AEMET, l'Agència Estatal de Meteorologia, publica les mesures diàries de les seves
estacions. Les nostres són de l'aeroport de Palma, de 2015 a 2023, i cada fila és un dia.
Els anys 2024 i 2025 els guardem a part: són el test segellat, i no els obrirem fins a la
UT10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Temps diari a l'aeroport de Palma (AEMET), sense els anys 2024 i 2025
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT04-Classificacio_logistica_knn/aemet/pluja_palma.csv"

MESURES = ["tmed", "tmin", "tmax", "prec", "velmedia", "racha", "sol", "presMax", "presMin"]

df = pd.read_csv(URL_DADES)
print(f"Dies: {len(df)}")
print(f"Dies que l'endemà plou: {df['plou_dema'].mean():.1%}")
df.head()

Les mesures de cada dia són les temperatures mitjana, mínima i màxima (`tmed`, `tmin`,
`tmax`, en °C), la pluja (`prec`, en mm), el vent mitjà i la ratxa més forta (`velmedia`,
`racha`, en m/s), les hores de sol (`sol`) i la pressió màxima i mínima (`presMax`,
`presMin`, en hPa).

La resposta és `plou_dema`: 1 si l'endemà plou i 0 si no. Volem predir **demà** amb el
que hem mesurat **avui**, que és el que podria fer servir una app del temps de debò. A
Palma, l'endemà plou un 13,1% dels dies, més o menys un de cada vuit. La columna
`pluja_forta_dema` la farem servir al NB 4.4.

## 2. El primer classificador de pluja

Seguim la recepta del NB 4.1, amb les nou mesures.

In [ ]:
X = df[MESURES]
y = df["plou_dema"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print(f"Encerts del model:     {model.score(X_test, y_test):.1%}")
print(f"Encerts de referència: {referencia.score(X_test, y_test):.1%}")

El model encerta un 87,2% dels dies. Sembla molt, però la referència, que diu sempre «no
plourà», en fa un 87,0%. Com que quasi mai no plou, dir que no és una aposta segura. **Amb
el percentatge d'encerts no podem saber si el model ha après res.** Ens cal mirar més a
dins.

## 3. La matriu de confusió

La **matriu de confusió** compta, per a cada classe real, què ha respost el model.

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=["no plou", "plou"], cmap="Blues"
)
plt.title("Regressió logística")
plt.show()

Les files (*True label*) són el que ha passat de veritat, i les columnes (*Predicted
label*), el que ha dit el model. Les
quatre caselles tenen nom:

| | El model diu «no plou» | El model diu «plou» |
|---|---|---|
| **No plou** | 708 **vertaders negatius** | 5 **falsos positius**: falses alarmes |
| **Plou** | 100 **falsos negatius**: pluges que se'ns escapen | 7 **vertaders positius** |

Positiu vol dir «la classe que busquem», aquí la pluja, i no «bona notícia». Ara es veu
el problema: dels 107 dies que va ploure, el model només n'ha avisat 7. Només diu «plou» 12
vegades en 820 dies.

---
### Ara tu · Exercici 1: llegir la matriu

*Uns 35 minuts, en grup.*

**a)** Sense ordinador, només amb els números de la matriu, responeu en forma de fracció
i de percentatge:

- Quin percentatge de dies encerta el model? Comproveu que surt el 87,2%.
- Quan el model diu «plou», quantes vegades acaba plovent?
- Dels dies que plou, quants n'avisa el model?
- Dels dies que no plou, quants encerta?

**b)** Dibuixeu la matriu de confusió de la referència, canviant `model` per `referencia`.
Quines caselles queden buides? Què respondria la referència a les preguntes de l'apartat a)?

**c)** Entreneu un k-NN amb `k=5` i dibuixeu-ne la matriu. Recordeu el NB 4.2: abans cal
escalar les mesures amb `StandardScaler`, i la matriu s'ha de fer amb les dades de prova
escalades. Té més o menys encerts que la regressió logística? I quantes pluges detecta?

**Compareu amb un altre grup.** Si organitzéssiu un concert a l'aire lliure, quin dels
tres models voldríeu? Per què el percentatge d'encerts no us ajuda a decidir-ho?

In [ ]:
# Escriu aquí el teu codi

## 4. Les mètriques, amb nom

Les preguntes que heu respost a mà tenen nom, i scikit-learn les calcula.

In [ ]:
prediccio = model.predict(X_test)

print(f"Accuracy:    {accuracy_score(y_test, prediccio):.1%}")
print(f"Precisió:    {precision_score(y_test, prediccio):.1%}")
print(f"Recall:      {recall_score(y_test, prediccio):.1%}")
print(f"Specificity: {recall_score(y_test, prediccio, pos_label=0):.1%}")

Totes surten de la matriu de confusió, i cadascuna respon una pregunta:

| Mètrica | Pregunta que respon | Aquí |
|---|---|---|
| **Accuracy** | Quin percentatge de dies encerta? | 87,2% |
| **Precisió** | Quan avisa de pluja, quantes vegades plou? | 58,3% |
| **Recall** | Dels dies que plou, quants n'avisa? | 6,5% |
| **Specificity** | Dels dies que no plou, quants n'encerta? | 99,3% |

Dues més surten restant: la **taxa de falsos positius** és 1 menys la specificity (aquí,
un 0,7% de falses alarmes) i la **taxa de falsos negatius** és 1 menys el recall (un 93,5%
de pluges que se'ns escapen). La specificity es calcula com un recall de la classe 0, i
per això hi posem `pos_label=0`.

Si n'heu de recordar dues, que siguin la **precisió** i el **recall**: diuen si les alarmes
són de fiar i si se n'escapen gaires. Aquest model, quan avisa, encerta més de la meitat de
les vegades, però quasi mai no avisa.

## 5. Moure el llindar

Com al NB 4.1, el model diu «plou» quan la probabilitat passa del 50%. Però la pluja
és tan rara que quasi mai no hi arriba. Provem llindars més baixos.

In [ ]:
probabilitat = model.predict_proba(X_test)[:, 1]

for llindar in [0.5, 0.3, 0.2, 0.1]:
    avisa = probabilitat >= llindar
    print(f"Llindar {llindar}: precisió {precision_score(y_test, avisa):.1%}, "
          f"recall {recall_score(y_test, avisa):.1%}")

Amb el llindar del 10%, el model ja avisa del 71,0% de les pluges, però només el 24,3% dels
seus avisos acaben en pluja. **Pujar el recall fa baixar la precisió**, i a l'inrevés: no
hi ha cap llindar que ho faci tot bé. Triar-lo és decidir quin error preferim, i això depèn
de qui farà servir el model.

---
### Ara tu · Exercici 2: tres clients

*Uns 35 minuts, en grup.*

Teniu tres clients que volen fer servir el vostre model de pluja. Per a cadascun:

- Digueu quin error li surt més car, una falsa alarma o una pluja que se li escapa, i
  quina mètrica ha de vigilar.
- Proveu llindars entre 0,05 i 0,6, i trieu-ne un.
- Dibuixeu la matriu de confusió del llindar triat amb aquesta línia:

```python
ConfusionMatrixDisplay.from_predictions(y_test, probabilitat >= 0.2,
                                        display_labels=["no plou", "plou"], cmap="Blues")
```

**Client A. Un càtering de noces a les finques de Mallorca.** Si plourà, munta una carpa.
Muntar-la quan no plou costa diners, però unes noces sense carpa sota la pluja són un
desastre.

**Client B. L'Ajuntament de Manacor.** Envia un SMS a tot el poble quan plourà. Si la gent
en rep massa d'equivocats, deixarà de fer-ne cas.

**Client C. Un pagès de Llevant.** Decideix cada vespre si rega l'endemà. Regar un dia
que plou malgasta aigua, i no regar un dia que no plou li fa perdre collita.

**Compareu amb un altre grup.** Heu triat el mateix llindar per a cada client? Hi ha
algun client a qui no serviria cap llindar?

In [ ]:
# Escriu aquí el teu codi

## 6. La corba ROC: tots els llindars alhora

Cada llindar dona una matriu de confusió diferent. La **corba ROC** les resumeix totes en un
sol dibuix, i així podem comparar models sense haver de triar abans el llindar.

In [ ]:
RocCurveDisplay.from_estimator(model, X_test, y_test, name="Regressió logística")
RocCurveDisplay.from_estimator(referencia, X_test, y_test, name="Referència",
                               ax=plt.gca(), linestyle="--")
plt.title("Corba ROC")
plt.show()

Cada punt de la corba és un llindar. A l'eix vertical hi ha el recall (*true positive
rate*) i a l'horitzontal, la taxa de falses alarmes (*false positive rate*). Un model
perfecte pujaria tot dret per l'esquerra fins a dalt; la referència, que no sap res, fa la
diagonal.

Per resumir-la en un número, es calcula l'**àrea sota la corba**, l'**AUC**: un 0,5 és
l'atzar i un 1, la perfecció. El nostre model fa 0,75. És lluny de la perfecció, però ha
après alguna cosa de debò, que és el que el percentatge d'encerts no ens deixava veure.
`ax=plt.gca()` serveix per dibuixar la segona corba al mateix gràfic que la primera.

---
### Ara tu · Exercici 3: el millor model per a Palma

*Uns 40 minuts, en grup.*

**a)** Dibuixeu al mateix gràfic les corbes ROC d'aquests models, i apunteu-ne l'AUC:

- la regressió logística amb les nou mesures, la de la secció 6;
- una regressió logística amb només dues mesures, triades per vosaltres;
- un k-NN amb les nou mesures escalades. Proveu `k=5`, `k=51` i `k=101`.

Recordeu que el k-NN s'ha de dibuixar amb les dades de prova escalades.

**b)** Algun k-NN pot tenir una AUC més alta que la regressió logística. Dibuixeu-ne la
matriu de confusió amb el llindar per defecte. Quantes vegades diu «plou»? Com és possible,
amb una AUC tan bona?

**c)** Trieu un dels tres clients de l'exercici 2 i doneu-li un model i un llindar. Escriviu-ne
la **fitxa** en quatre o cinc frases, sense paraules tècniques: «De cada 10 avisos, n'encerta...»,
«De cada 10 dies que plou, n'avisa...». Hi heu de dir també quin error se'n pot esperar.

**Compareu amb un altre grup.** Intercanvieu les fitxes. Entendria el client la vostra? Li
serviria?

In [ ]:
# Escriu aquí el teu codi

## 7. El que ens emportem

El percentatge d'encerts, sol, no basta quan una classe és rara. La **matriu de confusió**
mostra els quatre tipus de resposta, i d'aquí surten la **precisió** (les alarmes són de
fiar?) i el **recall** (se n'escapen gaires?). El **llindar** és la manera de triar entre
les dues segons el client, i la **corba ROC**, amb la seva AUC, compara models sense haver
de triar-lo.

Al NB 4.4 portarem aquesta idea a l'extrem amb la pluja forta, que només cau uns deu dies
l'any. Allà, un model que no serveix de res encertarà el 97% dels dies.